# Self-Attention from Scratch
### Query-Key-Value, the Matrix View, Multi-Head, and Positional Encoding

Self-attention is the core building block of the **Transformer** — the architecture behind BERT, GPT, and Vision Transformers. This notebook builds the mechanism **from scratch in NumPy** and then visualizes every part of it interactively.

## Why self-attention?

Imagine you want a layer that, for **every** word in a sentence, mixes in information from **every other** word.

- A **fully-connected / windowed (CNN-style) layer** only ever looks at a *fixed neighborhood*: a position can only see the few positions inside its window. To relate two far-apart words you must stack many layers.
- **Self-attention** lets *every position attend to every other position directly*, in a single layer, for a **variable-length** sequence. The layer learns, per pair of positions, *how much* one should attend to the other.

```
   windowed layer                 self-attention
   (fixed neighborhood)           (all-pairs, learned)

   a1  a2  a3  a4                 a1 - a2 - a3 - a4
    \  |  /                        \  X   X   X  /
     [win]                          all positions talk
```

## What we will build

1. **Scaled dot-product (QKV) attention** for a single output vector, step by step.
2. The **matrix formulation** that computes *all* outputs at once — and a numerical proof it matches the per-element version.
3. **Multi-head** attention, comparing what different heads attend to.
4. **Positional encoding**, showing that attention is permutation-equivariant and how sinusoidal encodings inject order.

## Learning objectives

- Explain *why* self-attention exists: a windowed layer sees only a fixed neighborhood, while self-attention lets every position attend to every other.
- Implement scaled dot-product attention from scratch for a single output vector (project to q/k/v, score, softmax, value-weighted sum).
- Re-express the computation in matrix form ($Q=W^q I$, $K=W^k I$, $V=W^v I$, $A=K^\top Q$, $A'=\mathrm{softmax}(A)$, $O=VA'$) and verify it matches numerically.
- Visualize the attention matrix $A'$ as a heatmap and see the effect of the $1/\sqrt{d_k}$ scaling.
- Extend to multi-head attention and compare per-head attention patterns.
- Show self-attention is permutation-equivariant and inject order with sinusoidal positional encoding.

> Everything runs in seconds on a **free Colab CPU runtime** — no downloads, no datasets, no API keys.

In [ ]:
# === Environment setup ===
# NumPy + matplotlib (+ ipywidgets for interactivity). Runs on a free Colab CPU runtime.
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

# ipywidgets is optional: under a headless verification kernel it may be missing.
# Degrade gracefully so the notebook still runs top-to-bottom.
try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except ImportError:
    widgets = None
    WIDGETS_AVAILABLE = False
    print("WARNING: ipywidgets not available \u2014 interactive controls disabled, "
          "default (non-interactive) plots will still render.")

# Legible default figure size for all plots.
plt.rcParams["figure.figsize"] = (6, 4)


def set_seed(seed: int = 0) -> int:
    """Seed NumPy's RNG so every cell shares the same random inputs/weights."""
    np.random.seed(seed)
    return seed


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically-stable softmax along `axis` (operates in float64)."""
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


# --- sanity checks (visible output under Run-all) ---
assert np.allclose(softmax(np.zeros((1, 4)), axis=-1), 0.25)
assert np.allclose(softmax(np.array([[1.0, 2.0, 3.0]]), axis=-1).sum(), 1.0)
print("Imports OK | numpy", np.__version__, "| matplotlib", matplotlib.__version__)
print("WIDGETS_AVAILABLE =", WIDGETS_AVAILABLE)

In [ ]:
# === Shared toy problem (single source of inputs + learned weights) ===
#
# MATRIX-SHAPE CONVENTION (fixed here, used everywhere):
#   * Inputs are stored COLUMN-WISE. I has shape [d, N]; column j is input vector a^j.
#   * Projections map d -> d_k (queries/keys) and d -> d_v (values).
#   * Q = W_q @ I, K = W_k @ I  have shape [d_k, N]   (column j = q^j / k^j)
#   * V = W_v @ I               has shape [d_v, N]     (column j = v^j)
#   * Attention matrix A = K.T @ Q has shape [N, N], softmaxed over axis=0 (keys),
#     so each QUERY column sums to 1. Output O = V @ A' has shape [d_v, N], column i = b^i.

set_seed(0)

TOKENS = ["the", "cat", "sat", "down"]
N = len(TOKENS)

d = 4      # model dimension (size of each input vector a^j)
d_k = 3    # query/key dimension
d_v = d_k  # value dimension (kept == d_k so single-head and multi-head O share shape)

# Shared input sequence: column j is the input vector for TOKENS[j].
I = np.random.randn(d, N).astype(np.float64)

# Learned projection matrices (scaled by 0.5 so q/k/v stay small and legible).
W_q = np.random.randn(d_k, d) * 0.5
W_k = np.random.randn(d_k, d) * 0.5
W_v = np.random.randn(d_v, d) * 0.5

# --- invariants ---
assert d_k <= d and d_v == d_k
assert I.shape == (d, N)
assert len(TOKENS) == N
assert W_q.shape == (d_k, d) and W_k.shape == (d_k, d) and W_v.shape == (d_v, d)

print("TOKENS :", TOKENS)
print("I.shape  =", I.shape, "(columns are input vectors a^j)")
print("W_q.shape=", W_q.shape, "| W_k.shape=", W_k.shape, "| W_v.shape=", W_v.shape)
print("d =", d, "| d_k =", d_k, "| d_v =", d_v, "| N =", N)

## 1. Scaled dot-product attention (Query, Key, Value)

For each input vector $a^i$ we produce three projections:

$$ q^i = W^q\, a^i \quad(\text{query}),\qquad k^i = W^k\, a^i \quad(\text{key}),\qquad v^i = W^v\, a^i \quad(\text{value}). $$

**Intuition:** the *query* of position $i$ asks "what am I looking for?", every *key* answers "here is what I offer", and the *value* is the information actually passed along.

**Score.** How much should position $i$ attend to position $j$? Take the dot product of $i$'s query with $j$'s key, then scale:

$$ \alpha_{i,j} = \frac{q^i \cdot k^j}{\sqrt{d_k}}. $$

The $1/\sqrt{d_k}$ factor keeps the scores from growing with the dimension $d_k$ (large scores push softmax toward a one-hot spike, which kills gradients).

**Normalize.** Softmax the scores over all keys $j$ so the attention weights for query $i$ form a probability distribution:

$$ \alpha'_{i,j} = \mathrm{softmax}_j(\alpha_{i,j}),\qquad \sum_j \alpha'_{i,j} = 1. $$

**Aggregate.** The output is the value-weighted sum:

$$ b^i = \sum_j \alpha'_{i,j}\, v^j. $$

> We use **dot-product** scoring (fast, just a matrix multiply). An alternative is *additive* attention, which feeds $q^i$ and $k^j$ through a small MLP instead. Dot-product is the standard choice in Transformers.

Next, we compute $b^1$ one number at a time.

In [ ]:
# === Compute b^1 step by step (0-indexed: query position 0 = 'the') ===

# (1) Project the whole sequence into queries, keys, values.
Q = W_q @ I   # [d_k, N], column j = q^j
K = W_k @ I   # [d_k, N], column j = k^j
V = W_v @ I   # [d_v, N], column j = v^j
assert Q.shape == (d_k, N) and V.shape == (d_v, N)

# (2) Take the query for position 1 (index 0).
q1 = Q[:, 0]                       # [d_k]

# (3) Raw scores: q1 dotted with every key k^j.
raw_scores = K.T @ q1              # [N]

# (4) Scale by 1/sqrt(d_k).
scale = np.sqrt(d_k)
assert scale > 0
scaled = raw_scores / scale       # [N]

# (5) Softmax over the N keys -> attention weights for query 1.
alpha1 = softmax(scaled, axis=-1) # [N], sums to 1

# (6) Output = value-weighted sum.
b1 = V @ alpha1                    # [d_v]

# --- checks ---
assert np.isclose(alpha1.sum(), 1.0)
assert np.all(alpha1 >= 0)
assert b1.shape == (d_v,)

# --- trace every number ---
np.set_printoptions(precision=4, suppress=True)
print("query q^1               :", np.round(q1, 4))
print("keys k^j (columns):\n", np.round(K, 4))
print()
for j, tok in enumerate(TOKENS):
    print(f"  j={j} ({tok:>4}) : raw={raw_scores[j]: .4f}  scaled={scaled[j]: .4f}  alpha'={alpha1[j]:.4f}")
print()
print("attention weights alpha'_{1,j} :", np.round(alpha1, 4), "(sum =", round(alpha1.sum(), 4), ")")
print("output b^1                      :", np.round(b1, 4))

In [ ]:
# === Interactive: attention weights for any query position i ===

def attention_for_position(i: int):
    """Return (alpha, b_i) for 0-based query position i over the shared Q, K, V."""
    assert 0 <= i < N, f"query position must be in [0, {N - 1}]"
    q_i = Q[:, i]
    scores = (K.T @ q_i) / np.sqrt(d_k)
    alpha = softmax(scores, axis=-1)
    b_i = V @ alpha
    return alpha, b_i


def plot_position(i: int) -> None:
    """Bar chart of attention weights for query position i, plus its output vector."""
    alpha, b_i = attention_for_position(i)
    fig, ax = plt.subplots()
    bars = ax.bar(range(N), alpha, color="#4C72B0")
    ax.set_xticks(range(N))
    ax.set_xticklabels(TOKENS)
    ax.set_ylim(0, 1)
    ax.set_xlabel("key position j (token)")
    ax.set_ylabel(r"$\alpha'_{i,j}$")
    ax.set_title(f"Attention weights for query position i={i} (\"{TOKENS[i]}\")")
    for rect, w in zip(bars, alpha):
        ax.text(rect.get_x() + rect.get_width() / 2, w + 0.02, f"{w:.2f}",
                ha="center", va="bottom", fontsize=9)
    plt.show()
    print(f"output b^{i} =", np.round(b_i, 4))


# --- default render so Run-all produces output before any interaction ---
plot_position(0)

# --- quick checks ---
alpha0, _ = attention_for_position(0)
assert np.isclose(alpha0.sum(), 1.0) and alpha0.shape == (N,)

# --- interactive control (only if ipywidgets is available) ---
if WIDGETS_AVAILABLE:
    slider = widgets.IntSlider(min=0, max=N - 1, value=0, description="query i")
    out = widgets.Output()

    def _update(change=None):
        with out:
            out.clear_output(wait=True)
            plot_position(slider.value)

    slider.observe(_update, names="value")
    _update()
    display(slider, out)

## 2. The matrix view: all outputs at once

Computing $b^i$ one position at a time is clear but slow. Because the inputs are stacked column-wise in $I$, **every** projection is a single matrix multiply:

$$ Q = W^q I,\qquad K = W^k I,\qquad V = W^v I. $$

The full set of all-pairs scores is one more matrix multiply (then scaled):

$$ A = \frac{K^\top Q}{\sqrt{d_k}}\quad\in\mathbb{R}^{N\times N}. $$

Here $A[j,i]$ is the score of key $j$ against query $i$, so the scores for query $i$ live in **column $i$**. We softmax **down each column** (over keys $j$):

$$ A' = \mathrm{softmax}_{\text{columns}}(A),\qquad \sum_j A'[j,i] = 1. $$

Finally, all outputs at once:

$$ O = V A'\quad\in\mathbb{R}^{d_v\times N},\qquad \text{column } i = b^i. $$

Two things to notice:

- Only $W^q, W^k, W^v$ are **learned**. Everything else is fixed matrix algebra.
- There is **no sequential dependency** between positions — the whole layer is one batch of matrix multiplies, which is exactly why it parallelizes so well on GPUs.

The next cell computes $A, A', O$ and *asserts* that column 0 of $O$ equals the $b^1$ we built by hand.

In [ ]:
# === Vectorized self-attention: ALL outputs in one pass ===

# (1) Projections (recomputed here so this cell is self-describing).
Q = W_q @ I   # [d_k, N]
K = W_k @ I   # [d_k, N]
V = W_v @ I   # [d_v, N]

# (2) Scaled all-pairs scores. A[j, i] = score of key j against query i -> query i in COLUMN i.
A = (K.T @ Q) / np.sqrt(d_k)      # [N, N]
assert A.shape == (N, N)

# (3) Softmax DOWN each column (over keys j) so every query column sums to 1.
A_prime = softmax(A, axis=0)     # [N, N]

# (4) All outputs at once. Column i of O is b^i.
O = V @ A_prime                  # [d_v, N]
assert O.shape == (d_v, N)

# (5) Equivalence with the per-element result from C05.
if not (np.allclose(O[:, 0], b1, atol=1e-8) and np.allclose(A_prime[:, 0], alpha1, atol=1e-8)):
    print("MISMATCH \u2014 convention bug:")
    print("  O[:,0]  =", np.round(O[:, 0], 6), " b1 =", np.round(b1, 6))
    print("  A'[:,0] =", np.round(A_prime[:, 0], 6), " alpha1 =", np.round(alpha1, 6))
assert np.allclose(A_prime.sum(axis=0), 1.0)
assert np.allclose(O[:, 0], b1, atol=1e-8)
assert np.allclose(A_prime[:, 0], alpha1, atol=1e-8)

# --- show the matrices ---
np.set_printoptions(precision=4, suppress=True)
print("A (scaled scores) =\n", np.round(A, 4))
print("\nA' (column-softmaxed, each query column sums to 1) =\n", np.round(A_prime, 4))
print("\ncolumn sums of A' :", np.round(A_prime.sum(axis=0), 6))
print("\nMatrix view matches per-element b^1:", True)

In [ ]:
# === Interactive: attention-matrix heatmap and the role of scaling/temperature ===

def recompute_scores() -> np.ndarray:
    """Raw (unscaled) all-pairs scores K^T Q, shape [N, N]."""
    return K.T @ Q


def plot_heatmap(apply_scaling: bool, temperature: float) -> None:
    """Render A' as a heatmap, optionally applying 1/sqrt(d_k) and a softmax temperature."""
    raw = recompute_scores()
    scores = raw / np.sqrt(d_k) if apply_scaling else raw.copy()
    scores = scores / max(temperature, 1e-6)
    A_vis = softmax(scores, axis=0)

    fig, ax = plt.subplots()
    im = ax.imshow(A_vis, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, label=r"$\alpha'$")
    ax.set_xticks(range(N)); ax.set_xticklabels(TOKENS)
    ax.set_yticks(range(N)); ax.set_yticklabels(TOKENS)
    ax.set_xlabel("query"); ax.set_ylabel("key")
    ax.set_title(f"Attention matrix A'  | scaling={apply_scaling}  temperature={temperature:.1f}")
    for jj in range(N):
        for ii in range(N):
            ax.text(ii, jj, f"{A_vis[jj, ii]:.2f}", ha="center", va="center",
                    color="white" if A_vis[jj, ii] < 0.5 else "black", fontsize=8)
    plt.show()
    print("mean max column weight:", round(float(A_vis.max(axis=0).mean()), 4),
          "(higher = sharper / more one-hot attention)")


# --- default render for Run-all ---
plot_heatmap(apply_scaling=True, temperature=1.0)

# --- show that scaling changes sharpness ---
_scaled = softmax((recompute_scores() / np.sqrt(d_k)), axis=0)
_unscaled = softmax(recompute_scores(), axis=0)
assert np.allclose(_scaled.sum(axis=0), 1.0)
print("mean max col weight  scaled:", round(float(_scaled.max(axis=0).mean()), 4),
      "| unscaled:", round(float(_unscaled.max(axis=0).mean()), 4))
print("Note: less scaling / lower temperature pushes attention toward one-hot.")

# --- interactive controls ---
if WIDGETS_AVAILABLE:
    cb = widgets.Checkbox(value=True, description="apply 1/sqrt(d_k)")
    temp = widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description="temperature")
    out = widgets.Output()

    def _update(change=None):
        with out:
            out.clear_output(wait=True)
            plot_heatmap(cb.value, temp.value)

    cb.observe(_update, names="value")
    temp.observe(_update, names="value")
    _update()
    display(widgets.HBox([cb, temp]), out)

## 3. Multi-head self-attention

A single attention pattern can only capture **one kind** of relationship. Multi-head attention runs several attention computations in parallel, each in its own subspace, so different heads can specialize in *different types of relevance* (e.g. one head tracks the adjacent word, another tracks a distant dependency).

Split each query/key/value into $h$ heads:

$$ q^i = [\,q^{i,1};\, q^{i,2};\, \dots;\, q^{i,h}\,],\qquad \text{(and likewise for } k^i, v^i\text{)}. $$

Each head computes scaled dot-product attention **independently** using only its own slice (scaled by $1/\sqrt{d_{\text{head}}}$ where $d_{\text{head}} = d_k/h$):

$$ b^{i,m} = \sum_j \mathrm{softmax}_j\!\left(\frac{q^{i,m}\cdot k^{j,m}}{\sqrt{d_{\text{head}}}}\right) v^{j,m}. $$

The per-head outputs are **concatenated** and projected by a learned output matrix $W^O$:

$$ b^i = W^O \big[\,b^{i,1};\, b^{i,2};\, \dots;\, b^{i,h}\,\big]. $$

In the next cell we implement $h$-head attention (default $h=2$) and plot each head's attention map side by side so you can see them attend differently.

In [ ]:
# === Multi-head self-attention (parameterizable head count h, default 2) ===

def multi_head_attention(h: int):
    """Return (per_head_A_prime, O_multi) for h heads over the shared Q, K, V."""
    if d_k % h != 0:
        valid = [x for x in range(1, d_k + 1) if d_k % x == 0]
        raise ValueError(f"h={h} must divide d_k={d_k}; valid head counts: {valid}")
    head_dim = d_k // h          # d_v == d_k so values split the same way

    per_head_A = []
    O_heads = []
    for m in range(h):
        sl = slice(m * head_dim, (m + 1) * head_dim)
        Q_h, K_h, V_h = Q[sl, :], K[sl, :], V[sl, :]
        A_h = (K_h.T @ Q_h) / np.sqrt(head_dim)   # [N, N]
        A_h_prime = softmax(A_h, axis=0)
        O_h = V_h @ A_h_prime                       # [head_dim, N]
        per_head_A.append(A_h_prime)
        O_heads.append(O_h)

    O_concat = np.concatenate(O_heads, axis=0)      # [d_v, N]
    set_seed(1)
    W_O = np.random.randn(d_v, d_v) * 0.5
    O_multi = W_O @ O_concat                         # [d_v, N]

    # --- checks ---
    for A_h_prime in per_head_A:
        assert np.allclose(A_h_prime.sum(axis=0), 1.0)
    assert O_multi.shape == (d_v, N) == O.shape
    assert len(per_head_A) == h
    return per_head_A, O_multi


def plot_heads(h: int) -> None:
    """Side-by-side per-head attention heatmaps."""
    per_head_A, O_multi = multi_head_attention(h)
    fig, axes = plt.subplots(1, h, figsize=(4 * h, 3.6), squeeze=False)
    for m, A_h_prime in enumerate(per_head_A):
        ax = axes[0, m]
        im = ax.imshow(A_h_prime, aspect="auto", cmap="viridis", vmin=0, vmax=1)
        ax.set_xticks(range(N)); ax.set_xticklabels(TOKENS, fontsize=8)
        ax.set_yticks(range(N)); ax.set_yticklabels(TOKENS, fontsize=8)
        ax.set_xlabel("query"); ax.set_ylabel("key")
        ax.set_title(f"head {m + 1}/{h}")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f"Per-head attention patterns (h={h} heads)")
    plt.tight_layout()
    plt.show()
    print("multi-head output shape:", O_multi.shape,
          "(matches single-head O:", O_multi.shape == O.shape, ")")


# --- default render for Run-all ---
plot_heads(2)

# --- interactive control: head counts that evenly divide d_k ---
if WIDGETS_AVAILABLE:
    divisors = [x for x in range(1, d_k + 1) if d_k % x == 0]
    default_h = 2 if (d_k % 2 == 0) else divisors[0]
    dd = widgets.Dropdown(options=divisors, value=default_h, description="# heads")
    out = widgets.Output()

    def _update(change=None):
        with out:
            out.clear_output(wait=True)
            plot_heads(dd.value)

    dd.observe(_update, names="value")
    _update()
    display(dd, out)

## 4. Positional encoding: giving attention a sense of order

Look back at the matrix formulation: nothing in $A = K^\top Q$ or $O = VA'$ depends on the *order* of the columns. If you shuffle the input positions, the outputs shuffle the **same way** but are otherwise unchanged. Self-attention is **permutation-equivariant** — it has no built-in notion of "first" vs "last".

That is a problem: "the cat sat down" and "down sat cat the" would produce the same set of representations. We fix it by adding a unique **positional vector** $e^i$ to each input *before* projection:

$$ a^i \;\leftarrow\; a^i + e^i. $$

The classic choice is **hand-crafted sinusoidal** encoding, where dimension $2i$ and $2i{+}1$ at position $\text{pos}$ use sine/cosine of geometrically-spaced frequencies:

$$ e_{2i}(\text{pos}) = \sin\!\Big(\frac{\text{pos}}{10000^{2i/d}}\Big),\qquad e_{2i+1}(\text{pos}) = \cos\!\Big(\frac{\text{pos}}{10000^{2i/d}}\Big). $$

Each position gets a distinct, smoothly-varying signature, and the pattern generalizes to lengths not seen in training. (Alternatives include **learned** positional embeddings and **relative** position encodings.)

The next cell builds the sinusoidal table, visualizes it, and demonstrates the equivariance break: shuffling the input leaves outputs unchanged-up-to-permutation **without** PE, but **with** PE the order genuinely matters.

In [ ]:
# === Sinusoidal positional encoding + permutation-equivariance demo ===

def positional_encoding(seq_len: int, d: int) -> np.ndarray:
    """Classic sinusoidal table, shape [d, seq_len]; column = position."""
    seq_len = max(int(seq_len), 1)
    pos = np.arange(seq_len)[None, :]                 # [1, seq_len]
    i2 = np.arange(0, d, 2)[:, None]                  # even dim indices -> [n_pairs, 1]
    div = np.power(10000.0, i2 / d)                   # [n_pairs, 1]
    PE = np.zeros((d, seq_len), dtype=np.float64)
    PE[0::2, :] = np.sin(pos / div)
    # guard odd d: there may be one fewer cos row than sin row
    n_cos = PE[1::2, :].shape[0]
    PE[1::2, :] = np.cos(pos / div[:n_cos])
    return PE


def run_attention(I_in: np.ndarray) -> np.ndarray:
    """Full self-attention for an input matrix I_in of shape [d, *]; returns O [d_v, *]."""
    Q_ = W_q @ I_in
    K_ = W_k @ I_in
    V_ = W_v @ I_in
    A_ = (K_.T @ Q_) / np.sqrt(d_k)
    A_p = softmax(A_, axis=0)
    return V_ @ A_p


# --- permutation-equivariance demonstration (reverse the sequence) ---
perm = np.arange(N)[::-1]

O_plain = run_attention(I)
equivariant = np.allclose(O_plain[:, perm], run_attention(I[:, perm]), atol=1e-8)

PE = positional_encoding(N, d)
I_pe = I + PE
order_aware = not np.allclose(run_attention(I_pe)[:, perm],
                              run_attention(I_pe[:, perm]), atol=1e-8)

assert positional_encoding(N, d).shape == (d, N)
assert equivariant       # WITHOUT PE: permuting input just permutes output
assert order_aware       # WITH PE: order now matters

print("Without PE  \u2014 permute(output) == output(permuted) ?", equivariant, "(permutation-equivariant)")
print("With PE     \u2014 output now depends on order ?       ", order_aware, "(order-aware)")


def plot_pe(seq_len: int, add_pe: bool) -> None:
    """Heatmap of the sinusoidal positional-encoding table."""
    table = positional_encoding(seq_len, d)
    fig, ax = plt.subplots()
    im = ax.imshow(table, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
    fig.colorbar(im, ax=ax, label="encoding value")
    ax.set_xlabel("position"); ax.set_ylabel("encoding dimension")
    state = "added to inputs" if add_pe else "shown only (not added)"
    ax.set_title(f"Sinusoidal positional encoding  (seq_len={seq_len}, {state})")
    plt.show()


# --- default render for Run-all ---
plot_pe(N, True)

# --- interactive controls ---
if WIDGETS_AVAILABLE:
    sl = widgets.IntSlider(min=2, max=16, value=N, description="seq_len")
    cb = widgets.Checkbox(value=True, description="add positional encoding")
    out = widgets.Output()

    def _update(change=None):
        with out:
            out.clear_output(wait=True)
            plot_pe(sl.value, cb.value)

    sl.observe(_update, names="value")
    cb.observe(_update, names="value")
    _update()
    display(widgets.HBox([sl, cb]), out)

## Wrap-up

We built the self-attention mechanism end to end:

1. **Per-element QKV attention** (C04–C06): projected each input to a query/key/value, scored $q^i\cdot k^j$, scaled by $1/\sqrt{d_k}$, softmaxed, and took a value-weighted sum to get $b^1$ — then inspected the attention weights for any query position.
2. **The matrix view** (C07–C09): recomputed *all* outputs at once with $Q=W^qI,\,K=W^kI,\,V=W^vI,\,A=K^\top Q/\sqrt{d_k},\,A'=\mathrm{softmax}(A),\,O=VA'$, **proved** it matches the per-element result, and visualized how $1/\sqrt{d_k}$ scaling controls attention sharpness.
3. **Multi-head attention** (C10–C11): split $Q/K/V$ into heads that attend independently, concatenated, and projected by $W^O$ — and saw heads form different patterns.
4. **Positional encoding** (C12–C13): showed self-attention is permutation-equivariant and injected order with sinusoidal encodings.

### Key takeaways

- Attention is a **softmax-weighted average of values**, where the weights come from query–key similarity.
- Only $W^q, W^k, W^v$ (and $W^O$ for multi-head) are **learned** — the rest is fixed matrix algebra.
- The matrix form is **fully parallel**: no position waits on another, which is why Transformers scale on GPUs.
- **Multiple heads** let the layer capture several relationships at once.
- Self-attention has **no notion of order** on its own; positions must be injected.

### Where this goes next

This toy NumPy implementation is *exactly* the primitive inside real Transformers:

- **BERT / RoBERTa** — bidirectional self-attention for language understanding.
- **GPT** — masked (causal) self-attention for generation.
- **ViT** — self-attention over image patches for vision.

### Further reading

- Hung-yi Lee, *Machine Learning (2021)* — Self-Attention lecture (source material).
- Vaswani et al., *Attention Is All You Need* (2017) — the original Transformer.
- J. Alammar, *The Illustrated Transformer* — a visual companion.